# IOAI — 2025 Selection Camp Angry Birds (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, urllib.request
os.makedirs('data/features', exist_ok=True)
_base = 'https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/angrybirds-features'
for f in ['train.pt', 'val.pt', 'test.pt']:
    if not os.path.exists(f'data/features/{f}'):
        urllib.request.urlretrieve(f'{_base}/{f}', f'data/features/{f}')
print('데이터 준비:', 'data/features/train.pt' if os.path.exists('data/features/train.pt') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Angry Birds — 모범답안 (Group DRO, 스퓨리어스 상관 견고)

> 데이터는 DGX에 **사전 준비**돼 있습니다 — `data/features/{train,val,test}.pt`(ResNet50 IMAGENET1K_V1 penultimate 2048-d 임베딩 + train/val 의 4-way 그룹 `y4`). 이미지 추출 없이 바로 분류기만 학습하면 됩니다(빠름).

**과제**: 새 종류(물새 0/뭍새 1)를 분류하되, 학습셋 뭍새에만 **빨간 사각형**(가짜 상관)이 섞여 있습니다. 지표 = 4개 그룹(새종류×배경) 중 **최저 정확도(worst-group acc)**(높을수록 좋음). held-out val 로 채점, 제출 `submission.csv`(datapointID, answer).

원대회 reference 방식: **그룹 가중 샘플링** + **Group DRO**(`RobustLoss`: 최악 그룹 손실 가중 + IRM 페널티)로 *진짜 새 특징*을 학습합니다. 베이스라인 worst-group≈0.71 → 모범답안≈0.80. (원본 전체 파이프라인(이미지→ResNet 추출)은 **모범답안2** 참고.)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, pandas as pd
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tr = torch.load('data/features/train.pt'); va = torch.load('data/features/val.pt'); te = torch.load('data/features/test.pt')
Xtr, y4tr = tr['X'], tr['y4']; Xva, y4va = va['X'], va['y4']

In [ ]:
class BirdsNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(nn.Dropout(0.6), nn.Linear(2048,32), nn.BatchNorm1d(32), nn.ReLU(), nn.Linear(32,2))
    def forward(self, x): return self.head(x)
class RobustLoss(nn.Module):   # Group DRO + IRM
    def __init__(self, irm_lambda=0.5, dro_eta=0.01):
        super().__init__(); self.il=irm_lambda; self.de=dro_eta; self.register_buffer('q', torch.ones(4)/4)
    def irm(self, lg, y):
        sc=torch.tensor(1.0, device=lg.device, requires_grad=True); l=F.cross_entropy(lg*sc, y)
        g=torch.autograd.grad(l, [sc], create_graph=True)[0]; return (g**2).sum()
    def forward(self, lg, y4):
        bl=y4//2; ps=F.cross_entropy(lg, bl, reduction='none'); gl=torch.zeros(4, device=lg.device); pen=0.
        for g in range(4):
            m=y4==g
            if m.any(): gl[g]=ps[m].mean(); pen=pen+self.irm(lg[m], bl[m])
        self.q=self.q*torch.exp(gl.detach()*self.de); self.q=self.q/self.q.sum()
        return torch.sum(self.q*gl) + self.il*pen

In [ ]:
counts=np.bincount(y4tr.numpy()); sw=torch.from_numpy((1.0/counts)[y4tr.numpy()])
dl=DataLoader(TensorDataset(Xtr,y4tr), batch_size=64, sampler=WeightedRandomSampler(sw, len(sw)))
torch.manual_seed(43); model=BirdsNet().to(dev); crit=RobustLoss().to(dev)
opt=torch.optim.AdamW(model.parameters(), lr=5e-6, weight_decay=0.05)
def worst_group():
    model.eval()
    with torch.no_grad(): pr=model(Xva.to(dev)).argmax(1).cpu()
    a=[ (pr[y4va==g]==(y4va[y4va==g]//2)).float().mean().item() for g in range(4) ]
    return min(a), pr
best=-1; best_pred=None
for ep in range(60):
    model.train()
    for xb,yb in dl:
        xb,yb=xb.to(dev),yb.to(dev); opt.zero_grad(); crit(model(xb),yb).backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
    wa,pr=worst_group()
    if wa>best: best=wa; best_pred=pr
    if ep%10==0 or ep==59: print(f'epoch {ep}: worst-group acc {wa:.4f}')
print('best worst-group acc:', round(best,4))

In [ ]:
# held-out val 예측 → submission.csv (우리 채점기용)
pd.DataFrame({'datapointID': va['id'], 'answer': best_pred.numpy()}).to_csv('submission.csv', index=False)
# test 예측 → submission_judge.csv (원대회 judge 형식)
model.eval()
with torch.no_grad(): tp = model(te['X'].to(dev)).argmax(1).cpu().numpy()
pd.DataFrame({'datapointID': te['id'], 'subtaskID': 1, 'answer': tp}).to_csv('submission_judge.csv', index=False)
print('wrote submission.csv (val) +', 'submission_judge.csv (test)', len(tp))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)